# 信息如何改善预测
先修：有限概率、加权平均与平方误差。先计算两组均值，再细分信息，最后观察选择如何改变独立性。
预测：知道市场状态以后，平均平方误差会增大还是减小？

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
x = np.array([0., 2., 1., 5., 4., 6., 7., 9.])
p = np.full(8, 1/8)  # 可改成不等概率；保持非负且总和为 1
z = np.repeat([0, 1], 4)
w = np.repeat(np.arange(4), 2)
assert np.all(p >= 0) and np.isclose(p.sum(), 1)
coarse = np.zeros(8)
for label in np.unique(z):
    group = z == label
    coarse[group] = np.sum(p[group]*x[group]) / p[group].sum()
fine = np.zeros(8)
for label in np.unique(w):
    group = w == label
    fine[group] = np.sum(p[group]*x[group]) / p[group].sum()
print('粗、细预测：', coarse, fine, sep='\n')

粗信息只能分两组，因此每组只能报同一个预测。不是计算机筛选定义了条件期望，而是信息允许的函数只能在组内为常数。最小化组内平方误差就得到加权均值。

In [ ]:
mean = p @ x
print('无信息、粗信息、细信息的均方误差：',
      p @ (x-mean)**2, p @ (x-coarse)**2, p @ (x-fine)**2)
for label in np.unique(z):
    group = z == label
    print('组', label, '残差加权和', p[group] @ (x-coarse)[group],
          '细预测的粗平均', p[group] @ fine[group] / p[group].sum())
within = p @ (x-coarse)**2
between = p @ (coarse-mean)**2
print('总方差 / 组内 + 组间：', p @ (x-mean)**2, within+between)
fig, ax = plt.subplots()
ax.plot(x, 'o', label='outcome'); ax.step(range(8), coarse, where='mid', label='coarse')
ax.step(range(8), fine, where='mid', label='fine'); ax.legend(); plt.show()

等概率时三项方差为 $135/16,54/16,81/16$。改成不等概率后数值改变，分解仍成立。尝试 `p = np.arange(1, 9)/36`，重新从数据格运行。塔式性质依赖细分割确实包含粗信息。

In [ ]:
prior = np.array([.7, .3])
likelihood = np.array([.1, .8])
evidence = prior @ likelihood
posterior = prior * likelihood / evidence
K = np.array([[.5, .5, 0], [0, .5, .5]])
print('信号概率、后验：', evidence, posterior)
print('混合分布：', np.array([.5, .5]) @ K)

## 选择能否制造相关
独立公平二元变量有四种等概率结果。仅观察两者恰好一个为 1 的结果，先预测剩下哪几行。

In [ ]:
states = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
selected = states[states.sum(axis=1) == 1]
for name, sample in [('全部', states), ('筛选后', selected)]:
    joint = np.mean((sample[:, 0] == 1) & (sample[:, 1] == 1))
    product = sample[:, 0].mean() * sample[:, 1].mean()
    print(name, sample.tolist(), '联合与边缘乘积：', joint, product)
# 共同原因：A=B=Z。给定 Z 后两者均为常数。
common = np.array([[0, 0], [1, 1]])
print('共同原因的边缘联合与乘积：', .5, .5*.5)

## 练习与反馈
为何细信息的误差不大于粗信息？粗预测也属于细信息允许的函数，扩大可选集合不会提高最小值。筛选后只剩 $(0,1),(1,0)$，知道一个变量便知道另一个；联合概率为 0 而边缘乘积为 $1/4$。共同原因例中边缘相关，给定原因后两变量都是常数，条件联合恰好等于条件边缘乘积。

延伸：将两组权重改成 1/4 与 3/4，先预测总体均值如何向第二组移动，再计算。